# Lab: Train an ML model on ERA5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ml_play/train_era5.ipynb)

Can a small neural network learn tomorrow's weather from today's? Here you train one on 10 years of daily ERA5 500 hPa geopotential height (Z500): it sees one day's map and predicts the next day's.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU): training takes about a minute there and much longer on a CPU. Run the two setup cells first.

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q gcsfs zarr cartopy torch

In [ ]:
#@title Setup: imports and ERA5 data
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
from torch import nn

# Daily means of ERA5 from 1959 to 2022 on a 1.5 degree grid (240 x 121 points).
# This is WeatherBench 2's copy on Google Cloud; no account needed.
era5 = xr.open_zarr("gs://weatherbench2/datasets/era5_daily/1959-2023_01_10-1h-240x121_equiangular_with_poles_conservative.zarr",
                    storage_options=dict(token="anon"))
era5 = era5.transpose(..., "latitude", "longitude").sortby("latitude")   # stored as (time, lon, lat); put latitude first, south to north

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
print("Training on", device)

---
## 1. The data

Z500 is the height of the 500 hPa pressure surface, about halfway up through the atmosphere's mass. Its highs (ridges) and lows (troughs) are the weather systems that move around the globe from day to day.

In [ ]:
# 10 years of daily Z500 (~3 GB to download, a few minutes). Geopotential / g = height in meters.
z500 = era5["geopotential"].sel(level=500, time=slice("2010", "2019")).compute() / 9.80665
print(f"{z500.sizes['time']} days, {z500.sizes['latitude']} x {z500.sizes['longitude']} grid points")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), layout="constrained", subplot_kw=dict(projection=ccrs.Robinson(central_longitude=180)))

for ax, day in zip(axes, ["2015-01-15", "2015-01-16"]):
    mesh = ax.pcolormesh(z500.longitude, z500.latitude, z500.sel(time=day), transform=ccrs.PlateCarree(), vmin=4900, vmax=5900)
    ax.coastlines(linewidth=0.5)
    ax.set_title(f"Z500 on {day}")
fig.colorbar(mesh, ax=axes, shrink=0.8, label="m")
plt.show()

The model's input is one day's map and its target is the next day's. Neural networks train best on numbers of order one, so the heights are standardized first (minus their mean, divided by their standard deviation).

A convolution applies the same filters everywhere on the map, so on its own the network cannot tell the tropics from the midlatitudes, where the weather moves the opposite way. A second input map, the same every day, tells it the latitude of each point: sin(latitude), which is also how the Coriolis effect varies.

In [ ]:
mean, std = float(z500.mean()), float(z500.std())
z = torch.tensor(((z500 - mean) / std).values, dtype=torch.float32)[:, None].to(device)   # (days, 1 channel, latitude, longitude)

# The latitude map: sin(latitude) at every grid point, shape (1, 1, latitude, longitude)
latitude = torch.tensor(np.sin(np.deg2rad(z500.latitude.values)), dtype=torch.float32).to(device)
latitude = latitude[None, None, :, None].expand(1, 1, z500.sizes["latitude"], z500.sizes["longitude"])

# A typical day-to-day change, so the network's output can be of order one too
change = float((z[1:] - z[:-1]).std())
print(f"{len(z)} days; a typical daily change is {change:.3f} standard deviations")

---
## 2. The model

A convolutional neural network (CNN): four layers of 5 x 5 filters, each looking at a point and its neighbors. Stacked, they see about 25° around each point, more than weather systems move in a day. The map is padded before each layer: around the globe in longitude ("circular"), so there is no edge at the date line, and by repeating the edge rows at the poles.

Tomorrow looks a lot like today, so the network predicts only the change, and the forecast is today plus that change.

The network will be asked to forecast many days ahead by feeding its forecasts back in, so it is trained that way too: from each starting day it steps forward ROLLOUT_STEPS days, and the loss adds up the error at every step. This teaches it not to let its own errors grow.

In [ ]:
# Four layers of 5 x 5 filters. Input: 2 maps (Z500 and latitude). Output: 1 map (the change in Z500).
model = nn.Sequential(
    nn.CircularPad2d((2, 2, 0, 0)), nn.ReplicationPad2d((0, 0, 2, 2)), nn.Conv2d(2, 64, kernel_size=5), nn.ReLU(),
    nn.CircularPad2d((2, 2, 0, 0)), nn.ReplicationPad2d((0, 0, 2, 2)), nn.Conv2d(64, 64, kernel_size=5), nn.ReLU(),
    nn.CircularPad2d((2, 2, 0, 0)), nn.ReplicationPad2d((0, 0, 2, 2)), nn.Conv2d(64, 64, kernel_size=5), nn.ReLU(),
    nn.CircularPad2d((2, 2, 0, 0)), nn.ReplicationPad2d((0, 0, 2, 2)), nn.Conv2d(64, 1, kernel_size=5),
).to(device)
print(f"{sum(p.numel() for p in model.parameters())} parameters")

EPOCHS = 10          # passes through all the days
BATCH_SIZE = 16      # starting days per training step
ROLLOUT_STEPS = 3    # days to step forward from each starting day while training
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(EPOCHS):
    order = torch.randperm(len(z) - ROLLOUT_STEPS)   # a new random order of the starting days each pass
    for i in range(0, len(order), BATCH_SIZE):
        start = order[i:i + BATCH_SIZE]
        state = z[start]
        loss = 0
        for k in range(1, ROLLOUT_STEPS + 1):
            inputs = torch.cat([state, latitude.expand(len(start), -1, -1, -1)], dim=1)   # Z500 and latitude
            state = state + change * model(inputs)                                       # the forecast for day k
            loss = loss + ((state - z[start + k]) ** 2).mean() / ROLLOUT_STEPS           # mean squared error
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f"epoch {epoch + 1}: loss {np.mean(losses[-len(order) // BATCH_SIZE:]):.5f}")

# The same loss for persistence: every day of the rollout = the starting day
persistence = np.mean([((z[k:] - z[:-k]) ** 2).mean().item() for k in range(1, ROLLOUT_STEPS + 1)])
plt.figure(figsize=(8, 3))
plt.plot(losses, color="black", linewidth=0.5, label="CNN")
plt.axhline(persistence, color="tab:red", label="every day = the starting day")
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("mean squared error")
plt.legend()
plt.show()

---
## 3. How good is it?

Compare the CNN with the simplest forecast of all, persistence: tomorrow will be the same as today. Errors are root mean square, in meters, weighted by area.

In [ ]:
# One-day forecasts from every day, 256 days at a time
forecast = []
with torch.no_grad():
    for i in range(0, len(z) - 1, 256):
        today = z[i:min(i + 256, len(z) - 1)]
        inputs = torch.cat([today, latitude.expand(len(today), -1, -1, -1)], dim=1)
        forecast.append(today + change * model(inputs))
forecast = torch.cat(forecast)[:, 0].cpu().numpy() * std + mean   # back to meters
today, tomorrow = z500.values[:-1], z500.values[1:]

area = np.cos(np.deg2rad(z500.latitude.values))[:, None] * np.ones(z500.sizes["longitude"])
rmse_cnn = np.sqrt(np.average(((forecast - tomorrow) ** 2).mean(axis=0), weights=area))
rmse_persistence = np.sqrt(np.average(((today - tomorrow) ** 2).mean(axis=0), weights=area))
print(f"RMSE of tomorrow = today: {rmse_persistence:.1f} m")
print(f"RMSE of the CNN:          {rmse_cnn:.1f} m")

Look at one day. The bottom row is the change from today to tomorrow: what actually happened, and what the CNN predicted.

In [ ]:
DAY = "2015-01-15"   # any day from 2010-01-01 to 2019-12-30

i = z500.indexes["time"].get_loc(DAY)
maps = [(today[i], f"Today, {DAY}"), (tomorrow[i], "Tomorrow: ERA5"), (forecast[i], "Tomorrow: CNN"),
        (tomorrow[i] - today[i], "Change: ERA5"), (forecast[i] - today[i], "Change: CNN")]
top = np.percentile(np.abs(tomorrow[i] - today[i]), 99)

fig = plt.figure(figsize=(16, 6.5), layout="constrained")
axes = []
for j, (values, title) in enumerate(maps):
    ax = fig.add_subplot(2, 3, j + 1 + (j >= 3), projection=ccrs.Robinson(central_longitude=180))
    if j < 3:
        mesh1 = ax.pcolormesh(z500.longitude, z500.latitude, values, transform=ccrs.PlateCarree(), vmin=4900, vmax=5900)
    else:
        mesh2 = ax.pcolormesh(z500.longitude, z500.latitude, values, transform=ccrs.PlateCarree(), cmap="RdBu_r", vmin=-top, vmax=top)
    ax.coastlines(linewidth=0.5)
    ax.set_title(title)
    axes.append(ax)
fig.colorbar(mesh1, ax=axes[:3], shrink=0.8, label="Z500 (m)")
fig.colorbar(mesh2, ax=axes[3:], shrink=0.8, label="change (m)")
plt.show()

---
## 4. Forecasting further ahead

The CNN only ever learned one day ahead. To go further, feed its forecast back in as the next day's input, over and over, like a weather model stepping forward in time. Pick a starting day and how many days to run.

In [ ]:
START = "2015-01-15"   # the starting day: 2010-01-01 to N_DAYS days before 2019-12-31
N_DAYS = 30            # how many days to forecast

# Start from ERA5 on START, then feed each forecast back in as the next day's input
i = z500.indexes["time"].get_loc(START)
state = z[i:i + 1]
cnn = []
with torch.no_grad():
    for day in range(N_DAYS):
        state = state + change * model(torch.cat([state, latitude], dim=1))
        cnn.append(state[0, 0].cpu().numpy() * std + mean)
cnn = np.array(cnn)
era5_days = z500.values[i + 1:i + 1 + N_DAYS]   # what actually happened
start_map = z500.values[i]

# Error at each lead time, compared with persistence (every day = the starting day)
rmse_cnn = np.sqrt(np.average((cnn - era5_days) ** 2, axis=(1, 2), weights=np.broadcast_to(area, cnn.shape)))
rmse_persistence = np.sqrt(np.average((start_map - era5_days) ** 2, axis=(1, 2), weights=np.broadcast_to(area, cnn.shape)))
days = np.arange(1, N_DAYS + 1)
plt.figure(figsize=(8, 3.5))
plt.plot(days, rmse_cnn, "o-", color="black", label="CNN")
plt.plot(days, rmse_persistence, "o-", color="tab:red", label="every day = the starting day")
plt.xlabel("days ahead")
plt.ylabel("RMSE (m)")
plt.title(f"Forecasts from {START}")
plt.legend()
plt.show()

Watch the forecast next to what actually happened. Where do they first part ways? What happens to the CNN's maps after a couple of weeks?

In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), layout="constrained", subplot_kw=dict(projection=ccrs.Robinson(central_longitude=180)))
for ax, title in zip(axes, ["ERA5", "CNN"]):
    ax.coastlines(linewidth=0.5)
    ax.set_title(title)

# One frame per day: an ERA5 map, a CNN map and a label, drawn in advance and shown one after another
frames = []
for d in range(N_DAYS):
    mesh = axes[0].pcolormesh(z500.longitude, z500.latitude, era5_days[d], transform=ccrs.PlateCarree(), vmin=4900, vmax=5900)
    mesh_cnn = axes[1].pcolormesh(z500.longitude, z500.latitude, cnn[d], transform=ccrs.PlateCarree(), vmin=4900, vmax=5900)
    label = axes[0].text(0, 1.08, f"day {d + 1}", transform=axes[0].transAxes, fontsize=12)
    frames.append([mesh, mesh_cnn, label])
fig.colorbar(mesh, ax=axes, shrink=0.8, label="Z500 (m)")

movie = animation.ArtistAnimation(fig, frames, interval=400)
plt.close(fig)   # show only the animation, not a still copy
HTML(movie.to_jshtml())

Things to try:
- Train for more epochs, or make the network wider (more than 64 filters) or deeper. Does the error keep going down?
- The model has only ever been graded on the days it trained on. Train on 2010–2018 and test on 2019: is it still better than persistence?
- Does the CNN do better or worse on a summer starting day than a winter one?
- Set ROLLOUT_STEPS = 1, or leave out the latitude map (an input of 1 map, not 2). How do the 30-day forecasts change?